# Task 4: Open-Set Recognition Launcher
Mount your Google Drive and set up the paths.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Base paths
DRIVE_ROOT = '/content/drive/MyDrive/atml_assignment1'
TASK4_DIR  = os.path.join(DRIVE_ROOT, 'task4')

# Download CIFAR to fast local Colab disk (not Drive)
DATA_DIR = '/content/cifar_data'
os.makedirs(DATA_DIR, exist_ok=True)

os.chdir(TASK4_DIR)
print(f'Working directory set to: {os.getcwd()}')

TASK4_CKPT_DIR  = os.path.join(TASK4_DIR, 'checkpoints')
TASK4_OUT_DIR   = os.path.join(TASK4_DIR, 'results')
TASK4_CACHE_DIR = os.path.join(TASK4_DIR, 'cache')

for d in [TASK4_CKPT_DIR, TASK4_OUT_DIR, TASK4_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Data will be downloaded to: {DATA_DIR}')
print(f'Checkpoints: {TASK4_CKPT_DIR}')
print(f'Results: {TASK4_OUT_DIR}')

### (Optional) Clean previous checkpoints and results

In [ ]:
import shutil

print('Cleaning up old Task 4 checkpoints, cache, and results...')
for d in [TASK4_CKPT_DIR, TASK4_OUT_DIR, TASK4_CACHE_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'  Deleted: {d}')
    os.makedirs(d, exist_ok=True)
print('Clean up complete!')

### Step 1a: Train Vanilla (100 epochs)

In [ ]:
!python train.py \
  --method vanilla \
  --data_root "{DATA_DIR}" \
  --checkpoints_dir "{TASK4_CKPT_DIR}" \
  --results_dir "{TASK4_OUT_DIR}"

### Step 1b: Train GCSC (100 epochs, same as Vanilla + RandAugment)

In [ ]:
!python train.py \
  --method gcsc \
  --data_root "{DATA_DIR}" \
  --checkpoints_dir "{TASK4_CKPT_DIR}" \
  --results_dir "{TASK4_OUT_DIR}"

### Step 1c: Train PROSER (50 epochs, fine-tuned from Vanilla checkpoint)

In [ ]:
!python train.py \
  --method proser \
  --data_root "{DATA_DIR}" \
  --checkpoints_dir "{TASK4_CKPT_DIR}" \
  --results_dir "{TASK4_OUT_DIR}" \
  --vanilla_ckpt "{TASK4_CKPT_DIR}/vanilla_best.pth"

### Step 2: Extract Logits and Features to Cache
This caches the model outputs so evaluation is instant and all scores use the exact same data.

In [ ]:
!python extract_outputs.py --method vanilla --data_root "{DATA_DIR}" --checkpoints_dir "{TASK4_CKPT_DIR}" --cache_dir "{TASK4_CACHE_DIR}"
!python extract_outputs.py --method gcsc --data_root "{DATA_DIR}" --checkpoints_dir "{TASK4_CKPT_DIR}" --cache_dir "{TASK4_CACHE_DIR}"
!python extract_outputs.py --method proser --data_root "{DATA_DIR}" --checkpoints_dir "{TASK4_CKPT_DIR}" --cache_dir "{TASK4_CACHE_DIR}"

### Step 3: Evaluate Open-Set Recognition
Generates Table 1, Table 2, score distribution plots, and failure analysis.

In [ ]:
!python evaluate_osr.py \
  --cache_dir "{TASK4_CACHE_DIR}" \
  --results_dir "{TASK4_OUT_DIR}" \
  --data_root "{DATA_DIR}"